##### Copyright 2020 The TensorFlow Authors.

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Simple audio recognition: Recognizing keywords

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/tutorials/audio/simple_audio">
    <img src="https://www.tensorflow.org/images/tf_logo_32px.png" />
    View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/audio/simple_audio.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/docs/blob/master/site/en/tutorials/audio/simple_audio.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
    View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/docs/site/en/tutorials/audio/simple_audio.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

This tutorial demonstrates how to preprocess audio files in the WAV format and build and train a basic [automatic speech recognition](https://en.wikipedia.org/wiki/Speech_recognition) (ASR) model for recognizing ten different words. You will use a portion of the [Speech Commands dataset](https://www.tensorflow.org/datasets/catalog/speech_commands) ([Warden, 2018](https://arxiv.org/abs/1804.03209)), which contains short (one-second or less) audio clips of commands, such as "down", "go", "left", "no", "right", "stop", "up" and "yes".

Real-world speech and audio recognition [systems](https://ai.googleblog.com/search/label/Speech%20Recognition) are complex. But, like [image classification with the MNIST dataset](../quickstart/beginner.ipynb), this tutorial should give you a basic understanding of the techniques involved.

## Setup

Import necessary modules and dependencies. You'll be using `tf.keras.utils.audio_dataset_from_directory` (introduced in TensorFlow 2.10), which helps generate audio classification datasets from directories of `.wav` files. You'll also need [seaborn](https://seaborn.pydata.org) for visualization in this tutorial.

In [ ]:
!pip install -U -q tensorflow tensorflow_datasets
!pip install ai_edge_litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━ 398.7/572.6 MB 209.5 MB/s eta 0:00:01

In [ ]:
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from IPython import display

# Set the seed value for experiment reproducibility.
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

## Import the mini Speech Commands dataset

To save time with data loading, you will be working with a smaller version of the Speech Commands dataset. The [original dataset](https://www.tensorflow.org/datasets/catalog/speech_commands) consists of over 105,000 audio files in the [WAV (Waveform) audio file format](https://www.aelius.com/njh/wavemetatools/doc/riffmci.pdf) of people saying 35 different words. This data was collected by Google and released under a CC BY license.

Download and extract the `mini_speech_commands.zip` file containing the smaller Speech Commands datasets with `tf.keras.utils.get_file`:

In [ ]:
DATASET_PATH = tf.keras.utils.get_file(
      'mini_speech_commands.zip',
      origin="http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip",
      extract=True,
      cache_dir='.', cache_subdir='data')
data_dir = pathlib.Path(DATASET_PATH) / 'mini_speech_commands'

### Generate Synthetic `_noise_` and `_silence_` Classes

We create synthetic WAV files so the model can learn to reject non-speech input.

In [ ]:
"""
generate_noise_silence.py — extended version (fixed)
500 files per class, 20 variants × 25 files each
All slice/broadcast bugs fixed with end = min(start + length, n)
"""

import wave, struct, random, math
import numpy as np
import scipy.signal
import pathlib

SAMPLE_RATE       = 16000
N_SAMPLES         = 16000
N_FILES           = 500

def _norm(arr, target_rms=0.1):
    rms = np.sqrt(np.mean(arr ** 2)) + 1e-9
    return arr * (target_rms / rms)

def _to_int16(arr):
    arr = np.clip(arr, -1.0, 1.0)
    return (arr * 32767).astype(np.int16).tolist()

def write_wav(path, samples_float, sample_rate=SAMPLE_RATE):
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    int16 = _to_int16(samples_float)
    with wave.open(str(path), 'w') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(b''.join(struct.pack('<h', s) for s in int16))


# ═══════════════════════════════════════════════
#  NOISE generators (20 variants)
# ═══════════════════════════════════════════════

def gen_white_noise(n=N_SAMPLES):
    return _norm(np.random.randn(n) * random.uniform(0.05, 1.0))

def gen_pink_noise(n=N_SAMPLES):
    white  = np.random.randn(n)
    freqs  = np.fft.rfftfreq(n); freqs[0] = 1e-6
    pink   = np.fft.irfft(np.fft.rfft(white) / np.sqrt(freqs), n=n)
    return _norm(pink)

def gen_brown_noise(n=N_SAMPLES):
    white  = np.random.randn(n)
    freqs  = np.fft.rfftfreq(n); freqs[0] = 1e-6
    brown  = np.fft.irfft(np.fft.rfft(white) / freqs, n=n)
    return _norm(brown)

def gen_bandpass_noise(n=N_SAMPLES):
    lo  = random.uniform(200, 1500)
    hi  = random.uniform(2000, 7000)
    sos = scipy.signal.butter(4, [lo, hi], btype='bandpass', fs=SAMPLE_RATE, output='sos')
    return _norm(scipy.signal.sosfilt(sos, np.random.randn(n)))

def gen_harmonic_hum(n=N_SAMPLES):
    t    = np.arange(n) / SAMPLE_RATE
    base = random.choice([50, 60])
    sig  = sum(1/k * np.sin(2*math.pi*base*k*t + random.uniform(0, 2*math.pi))
               for k in range(1, 8))
    return _norm(sig)

def gen_click_and_pop(n=N_SAMPLES):
    sig = np.random.randn(n) * 0.002
    for _ in range(random.randint(3, 20)):
        pos    = random.randint(0, n - 1)
        w      = random.randint(1, 8)
        end    = min(pos + w, n)               # ← fix: clamp to array bounds
        w      = end - pos
        sig[pos:end] += random.uniform(0.3, 1.0) * random.choice([-1, 1])
    return _norm(sig)

def gen_sine_sweep(n=N_SAMPLES):
    t   = np.arange(n) / SAMPLE_RATE
    sig = scipy.signal.chirp(t, f0=random.uniform(200, 800),
                              f1=random.uniform(1000, 4000), t1=1.0,
                              method=random.choice(['linear', 'logarithmic']))
    fade = int(0.05 * n)
    env  = np.ones(n)
    env[:fade]  = np.linspace(0, 1, fade)
    env[-fade:] = np.linspace(1, 0, fade)
    return _norm(sig * env)

def gen_pulse_train(n=N_SAMPLES):
    sig  = np.random.randn(n) * 0.002
    rate = random.uniform(2, 15)
    t    = 0.0
    while t < 1.0:
        pos = int(t * SAMPLE_RATE)
        w   = random.randint(8, 40)
        end = min(pos + w, n)                  # ← fix
        w   = end - pos
        amp = random.uniform(0.2, 0.8)
        if w > 0:
            sig[pos:end] += amp * np.exp(-np.arange(w) / (w * 0.3))
        t += max(1/rate + random.gauss(0, 0.3/rate), 0.02)
    return _norm(sig)

def gen_spectral_noise(n=N_SAMPLES):
    white    = np.random.randn(n)
    freqs    = np.fft.rfftfreq(n)
    ctrl_x   = np.linspace(0, 1, 10)
    ctrl_y   = np.random.rand(10) + 0.1
    envelope = np.interp(np.linspace(0, 1, len(freqs)), ctrl_x, ctrl_y)
    return _norm(np.fft.irfft(np.fft.rfft(white) * envelope, n=n))

def gen_amplitude_modulated(n=N_SAMPLES):
    t   = np.arange(n) / SAMPLE_RATE
    mod = 0.5 + 0.5 * np.sin(2*math.pi*random.uniform(1, 8)*t)
    return _norm(np.random.randn(n) * mod)

def gen_crowd_noise(n=N_SAMPLES):
    """Overlapping bandpass bursts — crowd / cafeteria."""
    sig = np.zeros(n)
    for _ in range(random.randint(5, 15)):
        lo    = random.uniform(200, 1000)
        hi    = min(lo + random.uniform(500, 2000), 7500)
        sos   = scipy.signal.butter(2, [lo, hi], btype='bandpass', fs=SAMPLE_RATE, output='sos')
        burst = scipy.signal.sosfilt(sos, np.random.randn(n))
        env   = np.zeros(n)
        start  = random.randint(0, n // 2)
        length = random.randint(n // 8, n // 2)
        end    = min(start + length, n)        # ← fix
        env[start:end] = 1.0
        sig += burst * env * random.uniform(0.1, 0.5)
    return _norm(sig)

def gen_keyboard_typing(n=N_SAMPLES):
    """Sparse transients — keyboard / tapping."""
    sig = np.random.randn(n) * 0.001
    for _ in range(random.randint(5, 20)):
        pos   = random.randint(0, n - 1)
        w     = random.randint(20, 80)
        end   = min(pos + w, n)               # ← fix
        w     = end - pos
        amp   = random.uniform(0.2, 0.8)
        if w > 0:
            decay = np.exp(-np.arange(w) / (w * 0.15))
            sig[pos:end] += amp * decay * np.random.choice([-1, 1])
    return _norm(sig)

def gen_wind_noise(n=N_SAMPLES):
    """Low-frequency turbulence — outdoor wind on mic."""
    sos = scipy.signal.butter(3, 300, btype='low', fs=SAMPLE_RATE, output='sos')
    sig = scipy.signal.sosfilt(sos, np.random.randn(n))
    for _ in range(random.randint(1, 4)):
        start  = random.randint(0, n - 1)
        length = random.randint(500, 2000)
        end    = min(start + length, n)        # ← fix
        sig[start:end] *= random.uniform(2, 5)
    return _norm(sig)

def gen_tv_radio(n=N_SAMPLES):
    """Speech-like modulated noise — distant TV/radio."""
    carrier = np.random.randn(n)
    sos     = scipy.signal.butter(4, [300, 3400], btype='bandpass', fs=SAMPLE_RATE, output='sos')
    carrier = scipy.signal.sosfilt(sos, carrier)
    t       = np.arange(n) / SAMPLE_RATE
    mod     = np.ones(n)
    for f in [2, 3, 5, 7]:
        mod += 0.3 * np.sin(2*math.pi*f*t + random.uniform(0, math.pi))
    return _norm(carrier * mod)

def gen_machinery(n=N_SAMPLES):
    """Periodic mechanical noise — motor / compressor."""
    t    = np.arange(n) / SAMPLE_RATE
    freq = random.uniform(50, 300)
    sig  = np.zeros(n)
    for k in range(1, 6):
        amp = random.uniform(0.1, 0.5) / k
        sig += amp * np.sin(2*math.pi*freq*k*t + random.uniform(0, 2*math.pi))
    sig += np.random.randn(n) * 0.05
    return _norm(sig)

def gen_rain(n=N_SAMPLES):
    """Dense random transients — rain on surface."""
    sig     = np.random.randn(n) * 0.01
    n_drops = random.randint(200, 800)
    for _ in range(n_drops):
        pos = random.randint(0, n - 1)
        w   = random.randint(3, 15)
        end = min(pos + w, n)                  # ← fix
        w   = end - pos
        amp = random.uniform(0.1, 0.5)
        if w > 0:
            sig[pos:end] += amp * np.exp(-np.arange(w) / (w * 0.2))
    return _norm(sig)

def gen_echo_noise(n=N_SAMPLES):
    """Noise with artificial echo — reverberant room."""
    sig    = gen_pink_noise(n)
    delays = [int(0.05*SAMPLE_RATE), int(0.1*SAMPLE_RATE), int(0.2*SAMPLE_RATE)]
    decays = [0.6, 0.4, 0.2]
    result = sig.copy()
    for d, a in zip(delays, decays):
        padded = np.zeros(n)
        padded[d:] = sig[:n-d] * a
        result += padded
    return _norm(result)

def gen_telephone_noise(n=N_SAMPLES):
    """300–3400 Hz bandlimited — telephone channel simulation."""
    sos = scipy.signal.butter(6, [300, 3400], btype='bandpass', fs=SAMPLE_RATE, output='sos')
    sig = scipy.signal.sosfilt(sos, np.random.randn(n))
    sig += np.random.randn(n) * 0.005
    return _norm(sig)

def gen_static_burst(n=N_SAMPLES):
    """Intermittent bursts — radio static / interference."""
    sig = np.random.randn(n) * 0.005
    for _ in range(random.randint(2, 8)):
        start  = random.randint(0, n - 1)
        length = random.randint(100, 1000)
        end    = min(start + length, n)        # ← fix: was causing broadcast error
        actual = end - start
        amp    = random.uniform(0.3, 1.0)
        sig[start:end] += np.random.randn(actual) * amp
    return _norm(sig)

def gen_music_like(n=N_SAMPLES):
    """Multiple harmonically related sinusoids — background music."""
    t      = np.arange(n) / SAMPLE_RATE
    root   = random.uniform(100, 400)
    ratios = [1, 1.25, 1.5, 2, 2.5, 3, 4]
    sig    = np.zeros(n)
    for r in ratios:
        amp   = random.uniform(0.1, 0.5)
        phase = random.uniform(0, 2*math.pi)
        vib   = 1 + 0.005 * np.sin(2*math.pi*5*t)
        sig  += amp * np.sin(2*math.pi*root*r*vib*t + phase)
    sig += gen_pink_noise(n) * 0.1
    return _norm(sig)


# ═══════════════════════════════════════════════
#  SILENCE generators (20 variants)
# ═══════════════════════════════════════════════

def gen_digital_silence(n=N_SAMPLES):
    return np.zeros(n)

def gen_dither_silence(n=N_SAMPLES):
    return _norm(np.random.randn(n), target_rms=random.uniform(0.0001, 0.002))

def gen_dc_offset_silence(n=N_SAMPLES):
    return np.clip(random.uniform(-0.01, 0.01) + np.random.randn(n)*0.001, -0.05, 0.05)

def gen_faint_hum_silence(n=N_SAMPLES):
    t   = np.arange(n) / SAMPLE_RATE
    amp = random.uniform(0.0005, 0.005)
    return amp * np.sin(2*math.pi*random.choice([50, 60])*t)

def gen_envelope_silence(n=N_SAMPLES):
    tau = random.uniform(0.02, 0.1)
    env = np.exp(-np.arange(n) / (tau * SAMPLE_RATE))
    return _norm(np.random.randn(n) * env, target_rms=random.uniform(0.0001, 0.003))

def gen_breath_silence(n=N_SAMPLES):
    sos = scipy.signal.butter(2, [50, 400], btype='bandpass', fs=SAMPLE_RATE, output='sos')
    return _norm(scipy.signal.sosfilt(sos, np.random.randn(n)),
                 target_rms=random.uniform(0.0005, 0.004))

def gen_room_tone(n=N_SAMPLES):
    return _norm(gen_pink_noise(n), target_rms=random.uniform(0.0005, 0.005))

def gen_intermittent_silence(n=N_SAMPLES):
    sig = np.random.randn(n) * 0.0005
    pos = random.randint(int(0.1*n), int(0.9*n))
    w   = random.randint(20, 80)
    end = min(pos + w, n)                      # ← fix
    w   = end - pos
    if w > 0:
        sig[pos:end] += random.uniform(0.005, 0.02) * np.exp(-np.arange(w) / (w * 0.2))
    return sig

def gen_clipped_silence(n=N_SAMPLES):
    return np.clip(np.random.randn(n)*0.0008, -0.003, 0.003)

def gen_quantised_silence(n=N_SAMPLES):
    return np.round(np.random.randn(n)*0.002*128) / 128

def gen_subsonic_silence(n=N_SAMPLES):
    """Very low frequency rumble below speech — building vibration."""
    sos = scipy.signal.butter(2, 30, btype='low', fs=SAMPLE_RATE, output='sos')
    sig = scipy.signal.sosfilt(sos, np.random.randn(n))
    return _norm(sig, target_rms=random.uniform(0.0003, 0.003))

def gen_thermal_noise(n=N_SAMPLES):
    """Gaussian noise at very low RMS — thermal noise floor."""
    return np.random.randn(n) * random.uniform(0.00005, 0.0003)

def gen_slow_fade_silence(n=N_SAMPLES):
    """Very slow amplitude variation — AGC hunting."""
    t   = np.arange(n) / SAMPLE_RATE
    env = 0.5 + 0.5 * np.sin(2*math.pi*random.uniform(0.3, 1.5)*t)
    return _norm(np.random.randn(n)*0.001*env, target_rms=0.001)

def gen_power_cycle_silence(n=N_SAMPLES):
    """50/60Hz + 2nd harmonic at very low level — power supply ripple."""
    t   = np.arange(n) / SAMPLE_RATE
    f   = random.choice([50, 60])
    amp = random.uniform(0.0002, 0.002)
    return amp * (np.sin(2*math.pi*f*t) + 0.3*np.sin(2*math.pi*2*f*t))

def gen_mic_handling_silence(n=N_SAMPLES):
    """Very low-frequency thump — mic being held still but vibrating."""
    sos = scipy.signal.butter(2, [5, 80], btype='bandpass', fs=SAMPLE_RATE, output='sos')
    sig = scipy.signal.sosfilt(sos, np.random.randn(n))
    return _norm(sig, target_rms=random.uniform(0.0005, 0.003))

def gen_sparse_crackle_silence(n=N_SAMPLES):
    """Very sparse low-amplitude crackles — old recording artefact."""
    sig = np.zeros(n)
    for _ in range(random.randint(1, 5)):
        pos = random.randint(0, n - 1)
        w   = random.randint(1, 4)
        end = min(pos + w, n)                  # ← fix
        sig[pos:end] = random.uniform(0.005, 0.02) * random.choice([-1, 1])
    return sig

def gen_high_freq_hiss_silence(n=N_SAMPLES):
    """High frequency tape hiss — above 4kHz only."""
    sos = scipy.signal.butter(3, 4000, btype='high', fs=SAMPLE_RATE, output='sos')
    sig = scipy.signal.sosfilt(sos, np.random.randn(n))
    return _norm(sig, target_rms=random.uniform(0.0003, 0.003))

def gen_flicker_silence(n=N_SAMPLES):
    """1/f amplitude modulation of tiny noise — electronic flicker."""
    white   = np.random.randn(n)
    freqs   = np.fft.rfftfreq(n); freqs[0] = 1e-6
    flicker = np.fft.irfft(np.fft.rfft(white) / np.sqrt(freqs), n=n)
    flicker = (flicker - flicker.min()) / (flicker.max() - flicker.min() + 1e-9)
    return np.random.randn(n) * 0.001 * flicker

def gen_dual_tone_silence(n=N_SAMPLES):
    """Two very faint tones — DTMF residual / test tone."""
    t   = np.arange(n) / SAMPLE_RATE
    amp = random.uniform(0.001, 0.008)
    f1  = random.choice([697, 770, 852, 941])
    f2  = random.choice([1209, 1336, 1477, 1633])
    return amp * (np.sin(2*math.pi*f1*t) + np.sin(2*math.pi*f2*t)) * 0.5

def gen_empty_room_silence(n=N_SAMPLES):
    """Pink noise + faint hum + dither — realistic empty room."""
    t   = np.arange(n) / SAMPLE_RATE
    sig = gen_pink_noise(n) * random.uniform(0.0005, 0.003)
    sig += random.uniform(0.0001, 0.001) * np.sin(2*math.pi*50*t)
    sig += np.random.randn(n) * 0.0001
    return sig


# ═══════════════════════════════════════════════
#  Variant tables
# ═══════════════════════════════════════════════

NOISE_VARIANTS = [
    ('white',       gen_white_noise),
    ('pink',        gen_pink_noise),
    ('brown',       gen_brown_noise),
    ('bandpass',    gen_bandpass_noise),
    ('hum',         gen_harmonic_hum),
    ('clicks',      gen_click_and_pop),
    ('sweep',       gen_sine_sweep),
    ('pulses',      gen_pulse_train),
    ('spectral',    gen_spectral_noise),
    ('am',          gen_amplitude_modulated),
    ('crowd',       gen_crowd_noise),
    ('keyboard',    gen_keyboard_typing),
    ('wind',        gen_wind_noise),
    ('tv_radio',    gen_tv_radio),
    ('machinery',   gen_machinery),
    ('rain',        gen_rain),
    ('echo',        gen_echo_noise),
    ('telephone',   gen_telephone_noise),
    ('static',      gen_static_burst),
    ('music',       gen_music_like),
]

SILENCE_VARIANTS = [
    ('digital',      gen_digital_silence),
    ('dither',       gen_dither_silence),
    ('dc',           gen_dc_offset_silence),
    ('faint_hum',    gen_faint_hum_silence),
    ('ringdown',     gen_envelope_silence),
    ('breath',       gen_breath_silence),
    ('roomtone',     gen_room_tone),
    ('intermittent', gen_intermittent_silence),
    ('clipped',      gen_clipped_silence),
    ('quantised',    gen_quantised_silence),
    ('subsonic',     gen_subsonic_silence),
    ('thermal',      gen_thermal_noise),
    ('slow_fade',    gen_slow_fade_silence),
    ('power_cycle',  gen_power_cycle_silence),
    ('handling',     gen_mic_handling_silence),
    ('crackle',      gen_sparse_crackle_silence),
    ('hiss',         gen_high_freq_hiss_silence),
    ('flicker',      gen_flicker_silence),
    ('dual_tone',    gen_dual_tone_silence),
    ('empty_room',   gen_empty_room_silence),
]

FILES_PER_VARIANT = N_FILES // len(NOISE_VARIANTS)               # 25 files per variant


def generate_class(out_dir, variants, files_per_variant, label):
    out_dir = pathlib.Path(out_dir)
    if out_dir.exists():
        existing = list(out_dir.glob('*.wav'))
        print(f"  '{label}' already exists ({len(existing)} files), skipping.")
        return
    print(f"Generating {label} ({len(variants)} variants × {files_per_variant} files)...")
    idx = 0
    for vname, fn in variants:
        for k in range(files_per_variant):
            sig   = fn(N_SAMPLES)
            scale = random.uniform(0.1, 1.0)
            sig   = np.clip(sig * scale, -1.0, 1.0)
            write_wav(out_dir / f'{label}_{vname}_{k:04d}.wav', sig)
            idx += 1
    print(f"  ✓ {idx} files → {out_dir}")


generate_class(data_dir / '_noise_',   NOISE_VARIANTS,   FILES_PER_VARIANT, 'noise')
generate_class(data_dir / '_silence_', SILENCE_VARIANTS, FILES_PER_VARIANT, 'silence')

print("\nDone:")
print(f"  _noise_   : {len(list((data_dir/'_noise_').glob('*.wav')))} WAV files")
print(f"  _silence_ : {len(list((data_dir/'_silence_').glob('*.wav')))} WAV files")

The dataset's audio clips are stored in eight folders corresponding to each speech command: `no`, `yes`, `down`, `go`, `left`, `up`, `right`, and `stop`:

In [ ]:
commands = np.array(tf.io.gfile.listdir(str(data_dir)))
commands = commands[(commands != 'README.md') & (commands != '.DS_Store')]
print('Commands (including synthetic classes):', commands)


Divided into directories this way, you can easily load the data using `keras.utils.audio_dataset_from_directory`.

The audio clips are 1 second or less at 16kHz. The `output_sequence_length=16000` pads the short ones to exactly 1 second (and would trim longer ones) so that they can be easily batched.

In [ ]:
train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=data_dir,
    batch_size=64,
    validation_split=0.2,
    seed=0,
    output_sequence_length=16000,
    subset='both')

label_names = np.array(train_ds.class_names)
print()
print("label names:", label_names)

The dataset now contains batches of audio clips and integer labels. The audio clips have a shape of `(batch, samples, channels)`.

In [ ]:
train_ds.element_spec

This dataset only contains single channel audio, so use the `tf.squeeze` function to drop the extra axis:

In [ ]:
def squeeze(audio, labels):
  audio = tf.squeeze(audio, axis=-1)
  return audio, labels

train_ds = train_ds.map(squeeze, tf.data.AUTOTUNE)
val_ds = val_ds.map(squeeze, tf.data.AUTOTUNE)

The `utils.audio_dataset_from_directory` function only returns up to two splits. It's a good idea to keep a test set separate from your validation set.
Ideally you'd keep it in a separate directory, but in this case you can use `Dataset.shard` to split the validation set into two halves. Note that iterating over **any** shard will load **all** the data, and only keep its fraction.

In [ ]:
test_ds = val_ds.shard(num_shards=2, index=0)
val_ds = val_ds.shard(num_shards=2, index=1)

In [ ]:
for example_audio, example_labels in train_ds.take(1):
  print(example_audio.shape)
  print(example_labels.shape)

Let's plot a few audio waveforms:

In [ ]:
label_names[[1,1,3,0]]

In [ ]:
plt.figure(figsize=(16, 10))
rows = 3
cols = 3
n = rows * cols
for i in range(n):
  plt.subplot(rows, cols, i+1)
  audio_signal = example_audio[i]
  plt.plot(audio_signal)
  plt.title(label_names[example_labels[i]])
  plt.yticks(np.arange(-1.2, 1.2, 0.2))
  plt.ylim([-1.1, 1.1])

## Convert waveforms to spectrograms

The waveforms in the dataset are represented in the time domain. Next, you'll transform the waveforms from the time-domain signals into the time-frequency-domain signals by computing the [short-time Fourier transform (STFT)](https://en.wikipedia.org/wiki/Short-time_Fourier_transform) to convert the waveforms to as [spectrograms](https://en.wikipedia.org/wiki/Spectrogram), which show frequency changes over time and can be represented as 2D images. You will feed the spectrogram images into your neural network to train the model.

A Fourier transform (`tf.signal.fft`) converts a signal to its component frequencies, but loses all time information. In comparison, STFT (`tf.signal.stft`) splits the signal into windows of time and runs a Fourier transform on each window, preserving some time information, and returning a 2D tensor that you can run standard convolutions on.

Create a utility function for converting waveforms to spectrograms:

- The waveforms need to be of the same length, so that when you convert them to spectrograms, the results have similar dimensions. This can be done by simply zero-padding the audio clips that are shorter than one second (using `tf.zeros`).
- When calling `tf.signal.stft`, choose the `frame_length` and `frame_step` parameters such that the generated spectrogram "image" is almost square. For more information on the STFT parameters choice, refer to [this Coursera video](https://www.coursera.org/lecture/audio-signal-processing/stft-2-tjEQe) on audio signal processing and STFT.
- The STFT produces an array of complex numbers representing magnitude and phase. However, in this tutorial you'll only use the magnitude, which you can derive by applying `tf.abs` on the output of `tf.signal.stft`.

### Microspeech-Compatible Spectrogram (30ms window)

Microspeech uses a **30ms window** at 16kHz = **480 samples** per frame.
We align STFT parameters to match exactly:
- `frame_length=480` (30ms)
- `frame_step=160` (10ms hop, standard microspeech stride)
- Log-mel filterbank with 40 bins (matches TFLite Micro AudioPreprocessor)

Training and inference **must use the same preprocessing** — this was the root cause of wrong detections.

In [ ]:
# ── Constants that match the official TFLM audio_preprocessor_int8.tflite ──
# These MUST match the C firmware — do NOT change them independently.
import math

SAMPLE_RATE  = 16000   # Hz
WINDOW_SIZE  = 480     # samples (30 ms at 16 kHz)  ← preprocessor input width
WINDOW_STRIDE = 320    # samples (20 ms hop)
NUM_MEL_BINS = 40      # features per window         ← preprocessor output width
NUM_FRAMES   = math.floor((SAMPLE_RATE - WINDOW_SIZE) / WINDOW_STRIDE) + 1  # 49

print(f'Spectrogram shape: ({NUM_FRAMES}, {NUM_MEL_BINS})')   # (49, 40)

# Alias kept for backward-compat with downstream cells
TIME_STEPS   = NUM_FRAMES


## Preprocessor — TFLM Signal Library interpreter

`audio_preprocessor_int8.tflite` is built from Signal Library custom ops
(`SignalWindow`, `SignalRfft`, `SignalFilterBank`).  These ops **only exist**
in the TFLM C++ runtime — the standard `tf.lite.Interpreter` cannot load this
model (you will get `RuntimeError: Encountered unresolved custom op: SignalWindow`).

We use the **`tflite-micro`** Python package which ships the exact same Signal
Library kernels used by the MCU firmware.  This gives bit-exact feature parity
between training and deployment.

```
pip install tflite-micro
```

Two interpreters are allocated once and reused throughout the notebook:

| Name | Model file | Input dtype | Output dtype |
|---|---|---|---|
| `interp_pre_i8` | `audio_preprocessor_int8.tflite` | `int16 (1,480)` | `int8 (1,40)` |
| `interp_pre_f32` | `audio_preprocessor_float32.tflite` | `int16 (1,480)` | `float32 (1,40)` |

> **Note:** if you haven't exported these files yet, jump to the *Export*
> section first, then come back here.


In [ ]:
!rm -rf zephyr_tflm_v2_speech
!git clone --depth 1 --recurse-submodules https://github.com/williamchand/zephyr_tflm_v2_speech

In [ ]:
# ── install tflite-micro Python package (ships Signal Library kernels) ──────
!pip install -q tflite-micro

from tflite_micro import runtime as tflm_runtime
import numpy as np

# ── download preprocessor models from the repo if not already present ────────
import os, urllib.request

# ── allocate TFLM interpreters (Signal Library kernels included) ─────────────
interp_pre_i8  = tflm_runtime.Interpreter.from_file('/kaggle/working/zephyr_tflm_v2_speech/tflite_micro/tensorflow/lite/micro/examples/micro_speech/models/audio_preprocessor_int8.tflite')
interp_pre_f32 = tflm_runtime.Interpreter.from_file('/kaggle/working/zephyr_tflm_v2_speech/tflite_micro/tensorflow/lite/micro/examples/micro_speech/models/audio_preprocessor_float.tflite')

def _show_io(name, interp):
    i = interp.get_input_details(0)
    o = interp.get_output_details(0)
    print(f'  {name:<30}  in={i["dtype"].__name__}{list(i["shape"])}  '
          f'out={o["dtype"].__name__}{list(o["shape"])}')

print('Preprocessor I/O:')
_show_io('int8  (int16→int8)',    interp_pre_i8)
_show_io('float32 (int16→f32)',  interp_pre_f32)


## `get_spectrogram` — replaced by TFLM interpreter

All Python DSP code (`apply_hann_window`, `fft_auto_scale`, `compute_rfft`,
`compute_energy`, `apply_filter_bank`, `filter_bank_sqrt`, `spectral_subtraction`,
`pcan_gain_control`, `filter_bank_log`, `to_int8`) is **removed**.

`get_spectrogram()` now simply calls the TFLM interpreter.  It produces
bit-exact output to the C firmware because it runs the **same binary model**.

| `quantize=` | Interpreter | Frame in | Feature out |
|---|---|---|---|
| `False` | `interp_pre_f32` | `int16 (1, 480)` | `float32 (1, 40)` |
| `True` | `interp_pre_i8` | `int16 (1, 480)` | `int8 (1, 40)` |


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# get_spectrogram — calls TFLM interpreter, no Python DSP
# ═══════════════════════════════════════════════════════════════════════════

def get_spectrogram(audio_frame_int16: np.ndarray,
                    quantize: bool = False) -> np.ndarray:
    """
    Run one 480-sample window through the TFLM preprocessor.

    Args:
        audio_frame_int16 : int16 ndarray, shape (1, 480)  — raw PCM window
        quantize          : False → float32 (1, 40)  ← training
                            True  → int8   (1, 40)   ← MCU-match

    Returns:
        ndarray (1, 40), dtype float32 or int8
    """
    assert audio_frame_int16.dtype == np.int16,         f'Expected int16, got {audio_frame_int16.dtype}'
    assert audio_frame_int16.shape == (1, WINDOW_SIZE),         f'Expected (1, {WINDOW_SIZE}), got {audio_frame_int16.shape}'

    interp = interp_pre_i8 if quantize else interp_pre_f32
    interp.set_input(audio_frame_int16, 0)
    interp.invoke()
    return interp.get_output(0).copy()   # (1, 40) int8 or float32


# ═══════════════════════════════════════════════════════════════════════════
# _process_single — pads/trims, frames, calls get_spectrogram per frame
# ═══════════════════════════════════════════════════════════════════════════

def _process_single(waveform_np: np.ndarray,
                    quantize: bool = False) -> np.ndarray:
    """
    Convert a 1-second float32 waveform [-1, 1] to a spectrogram.

    Returns:
        quantize=False → (49, 40) float32   ← training default
        quantize=True  → (49, 40) int8      ← MCU match
    """
    # ── normalise input ──────────────────────────────────────────────────────
    wf = np.squeeze(waveform_np).astype(np.float32)

    if len(wf) < SAMPLE_RATE:
        wf = np.pad(wf, (0, SAMPLE_RATE - len(wf)))
    else:
        wf = wf[:SAMPLE_RATE]

    # ── convert to int16 range (matches MCU ADC — single cast, no round-trip) ─
    wf_i16 = np.clip(wf * 32767.0, -32768, 32767).astype(np.int16)

    # ── frame + run preprocessor ─────────────────────────────────────────────
    frames = []
    for start in range(0, SAMPLE_RATE - WINDOW_SIZE + 1, WINDOW_STRIDE):
        frame = wf_i16[start : start + WINDOW_SIZE].reshape(1, WINDOW_SIZE)
        feat  = get_spectrogram(frame, quantize=quantize)   # (1, 40)
        frames.append(feat.flatten())

    spec = np.stack(frames, axis=0)    # (49, 40) float32 or int8
    assert spec.shape == (NUM_FRAMES, NUM_MEL_BINS),         f'Bad shape: {spec.shape}'
    return spec


# ═══════════════════════════════════════════════════════════════════════════
# waveform_to_spectrogram — public API, unchanged signature
# ═══════════════════════════════════════════════════════════════════════════

def waveform_to_spectrogram(waveform, quantize: bool = False):
    """
    Args:
        waveform : (16000,) float32 [-1,1]  Tensor or ndarray
        quantize : False → (49, 40) float32   ← training
                   True  → (49, 40) int8      ← MCU match
    """
    if hasattr(waveform, 'numpy'):
        waveform = waveform.numpy()
    return _process_single(np.asarray(waveform, dtype=np.float32), quantize)


# ═══════════════════════════════════════════════════════════════════════════
# WAV helpers — unchanged API
# ═══════════════════════════════════════════════════════════════════════════

def wav_to_waveform(file_path, quantize: bool = False):
    """
    quantize=False → (16000,) float32 [-1, 1]
    quantize=True  → (16000,) int16
    """
    raw      = tf.io.read_file(file_path)
    wf, _    = tf.audio.decode_wav(raw, desired_channels=1,
                                   desired_samples=SAMPLE_RATE)
    wf       = tf.squeeze(wf, axis=-1)          # float32 [-1, 1]
    if quantize:
        return tf.cast(
            tf.clip_by_value(wf * 32767.0, -32768, 32767), tf.int16)
    return wf

def wav_to_spectrogram(file_path, quantize: bool = False):
    """(49, 40) float32 or int8"""
    wf = wav_to_waveform(file_path, quantize=False)   # always float32
    return waveform_to_spectrogram(wf, quantize=quantize)


In [ ]:
# ── sanity check: run one real file through both paths ───────────────────────
_wav = str(data_dir / 'no/01bb6a2a_nohash_0.wav')

spec_f32 = wav_to_spectrogram(_wav, quantize=False)   # (49, 40) float32
spec_i8  = wav_to_spectrogram(_wav, quantize=True)    # (49, 40) int8

print(f'float32  shape={spec_f32.shape}  dtype={spec_f32.dtype}  '
      f'min={spec_f32.min():.3f}  max={spec_f32.max():.3f}')
print(f'int8     shape={spec_i8.shape}   dtype={spec_i8.dtype}   '
      f'min={spec_i8.min()}  max={spec_i8.max()}')
print()
print('✅ get_spectrogram is now a thin wrapper around the TFLM interpreter.')
print('   No Python DSP.  Bit-exact match with C firmware.')


Next, start exploring the data. Print the shapes of one example's tensorized waveform and the corresponding spectrogram, and play the original audio:

In [ ]:
for i in range(3):
  label = label_names[example_labels[i]]
  waveform = example_audio[i]
  print(waveform)
  spectrogram = waveform_to_spectrogram(waveform)

  print('Label:', label)
  print('Waveform shape:', waveform.shape)
  print('Spectrogram shape:', spectrogram.shape)
  print('Audio playback')
  display.display(display.Audio(waveform, rate=16000))

Now, define a function for displaying a spectrogram:

In [ ]:
def plot_spectrogram(spectrogram, ax):
  if len(spectrogram.shape) > 2:
    assert len(spectrogram.shape) == 3
    spectrogram = np.squeeze(spectrogram, axis=-1)
  # Convert the frequencies to log scale and transpose, so that the time is
  # represented on the x-axis (columns).
  # Add an epsilon to avoid taking a log of zero.
  log_spec = np.log(spectrogram.T + np.finfo(float).eps)
  height = log_spec.shape[0]
  width = log_spec.shape[1]
  X = np.linspace(0, np.size(spectrogram), num=width, dtype=int)
  Y = range(height)
  ax.pcolormesh(X, Y, log_spec)

Plot the example's waveform over time and the corresponding spectrogram (frequencies over time):

In [ ]:
# Option 1: Remove .numpy() calls entirely
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
timescale = np.arange(waveform.shape[0])
axes[0].plot(timescale, waveform)
axes[0].set_title('Waveform')
axes[0].set_xlim([0, 16000])

plot_spectrogram(spectrogram, axes[1])
axes[1].set_title('Spectrogram')
plt.suptitle(label.title())
plt.tight_layout()
plt.show()

Now, create spectrogram datasets from the audio datasets:

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RECOMMENDED: Balanced augmentation pipeline
# ═══════════════════════════════════════════════════════════════════════════

# More aggressive waveform augmentation
def augment_waveform(waveform, label):
    # Volume variation (70-130%)
    gain = tf.random.uniform([], 0.7, 1.3)
    waveform = waveform * gain
    waveform = tf.clip_by_value(waveform, -1.0, 1.0)
    
    # Background noise
    noise_std = tf.random.uniform([], 0.002, 0.01)
    noise = tf.random.normal(tf.shape(waveform), stddev=noise_std)
    waveform = waveform + noise
    waveform = tf.clip_by_value(waveform, -1.0, 1.0)
    
    # Time shift
    shift = tf.random.uniform([], -2400, 2400, dtype=tf.int32)
    waveform = tf.roll(waveform, shift, axis=0)
    
    return waveform, label


In [ ]:
BATCH_SIZE = 64

# ── tf.data wrapper: numpy_function bridges TF graph ↔ TFLM interpreter ─────
def _spec_numpy_fn(waveform_np):
    """Called by tf.numpy_function.  Always returns float32 for training."""
    return _process_single(waveform_np.astype(np.float32), quantize=False)

def _spec_tf(waveform_tensor):
    """tf.data-compatible wrapper around get_spectrogram."""
    spec = tf.numpy_function(
        func=_spec_numpy_fn,
        inp=[waveform_tensor],
        Tout=tf.float32,
    )
    spec.set_shape([NUM_FRAMES, NUM_MEL_BINS])   # required for Keras shape inference
    return spec

def make_spec_ds(ds, training=False):
    """
    Build spectrogram dataset.
    Preprocessing runs through TFLM interpreter (num_parallel_calls=1 —
    tflm_runtime interpreter is not thread-safe).
    """
    ds = ds.unbatch()

    if training:
        ds = ds.map(augment_waveform, num_parallel_calls=tf.data.AUTOTUNE)

    # ── TFLM preprocessor ────────────────────────────────────────────────────
    ds = ds.map(
        lambda audio, label: (_spec_tf(audio), label),
        num_parallel_calls=1,   # ← must be 1: tflm interpreter is not thread-safe
    )
    ds = ds.cache()

    if training:
        ds = ds.shuffle(10000)

    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
train_spectrogram_ds = make_spec_ds(train_ds, training=True)
val_spectrogram_ds = make_spec_ds(val_ds)
test_spectrogram_ds = make_spec_ds(test_ds)

Examine the spectrograms for different examples of the dataset:

In [ ]:
for example_spectrograms, example_spect_labels in train_spectrogram_ds.take(1):
  break

In [ ]:
rows = 3
cols = 3
n = rows*cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 9))

for i in range(n):
    r = i // cols
    c = i % cols
    ax = axes[r][c]
    plot_spectrogram(example_spectrograms[i].numpy(), ax)
    ax.set_title(label_names[example_spect_labels[i].numpy()])

plt.show()

## Build and train the model

Add `Dataset.cache` and `Dataset.prefetch` operations to reduce read latency while training the model:

In [ ]:
train_spectrogram_ds = train_spectrogram_ds.cache().shuffle(10000).prefetch(tf.data.AUTOTUNE)
val_spectrogram_ds = val_spectrogram_ds.cache().prefetch(tf.data.AUTOTUNE)
test_spectrogram_ds = test_spectrogram_ds.cache().prefetch(tf.data.AUTOTUNE)

## Model — TinyConv

The **TinyConv** architecture matches the official TFLM `micro_speech` reference
model exactly.  The C code in your repo (`micro_speech_test.cc`) already registers
the ops for this model: `DepthwiseConv2D`, `FullyConnected`, `Softmax`, `Reshape`.

| | TinyConv | Previous 4-block CNN |
|---|---|---|
| Params | ~8 K | ~61 K |
| int8 size | ~18 KB | ~71 KB |
| Registered C ops | `Conv2D`, `FC`, `Reshape` | same + more |

Input shape `(49, 40, 1)` — matches the `(NUM_FRAMES, NUM_MEL_BINS)` spectrogram
produced by the TFLM preprocessor above.


In [ ]:
from tensorflow.keras import layers, models

input_shape = example_spectrograms.shape[1:]   # (49, 40)
num_labels  = len(label_names)
print(f'Input shape: {input_shape}')
print(f'Labels ({num_labels}): {label_names}')

# ═══════════════════════════════════════════════════════════════════════════
# TinyConv Enhanced — NO REGULARIZATION, pure MCU-safe overfitting fixes
# ═══════════════════════════════════════════════════════════════════════════

model = models.Sequential([
    layers.Input(shape=input_shape),                          # (49, 40)
    layers.Reshape((input_shape[0], input_shape[1], 1)),      # (49, 40, 1)
    layers.Conv2D(
        filters=10,           # ← Reduced from 12
        kernel_size=(8, 10),
        strides=(2, 2),
        padding='same',
        use_bias=True,
        activation='relu',
        name='conv1',
    ),                                                        # (25, 20, 10)
    layers.Dropout(0.5, name='dropout1'),                     # ← Higher dropout
    layers.Conv2D(
        filters=12,           # ← Reduced from 16
        kernel_size=(4, 4),
        strides=(2, 2),       # ← Changed from (1,1) - downsample more
        padding='same',
        use_bias=True,
        activation='relu',
        name='conv2',
    ),                                                        # (13, 10, 12)
    layers.Dropout(0.5, name='dropout2'),                     # ← Add second dropout
    layers.Flatten(name='flatten'),                           # (1560,)
    layers.Dense(num_labels, name='fc'),                      # (num_labels,)
], name='tinyconv_2layer_compact')

model.summary()
print(f'\nTotal params: {model.count_params():,}')

Configure the Keras model with the Adam optimizer and the cross-entropy loss:

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)


Train the model over 10 epochs for demonstration purposes:

In [ ]:
EPOCHS = 200

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        verbose=1,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        verbose=1,
        min_lr=1e-5
    ),
]
history = model.fit(
    train_spectrogram_ds,
    validation_data=val_spectrogram_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

Let's plot the training and validation loss curves to check how your model has improved during training:

In [ ]:
metrics = history.history
plt.figure(figsize=(16,6))
plt.subplot(1,2,1)
plt.plot(history.epoch, metrics['loss'], metrics['val_loss'])
plt.legend(['loss', 'val_loss'])
plt.ylim([0, max(plt.ylim())])
plt.xlabel('Epoch')
plt.ylabel('Loss [CrossEntropy]')

plt.subplot(1,2,2)
plt.plot(history.epoch, 100*np.array(metrics['accuracy']), 100*np.array(metrics['val_accuracy']))
plt.legend(['accuracy', 'val_accuracy'])
plt.ylim([0, 100])
plt.xlabel('Epoch')
plt.ylabel('Accuracy [%]')

## Evaluate the model performance

Run the model on the test set and check the model's performance:

In [ ]:
model.evaluate(test_spectrogram_ds, return_dict=True)

## Confidence-Threshold Inference

After training, run inference with a confidence threshold. Predictions below the threshold are flagged as uncertain (could be noise or silence).

In [ ]:
CONFIDENCE_THRESHOLD = 0.7

def run_inference_with_threshold(file_path, model, label_names,
                                  threshold=CONFIDENCE_THRESHOLD):
    """WAV file → label, using TFLM preprocessor for feature extraction."""
    raw      = tf.io.read_file(str(file_path))
    wf, _    = tf.audio.decode_wav(raw, desired_channels=1)
    waveform = tf.squeeze(wf, axis=-1)                        # float32 [-1,1]

    spec    = waveform_to_spectrogram(waveform, quantize=False)   # (49,40) float32
    spec_in = tf.constant(spec[np.newaxis, ...], dtype=tf.float32)  # (1,49,40)

    logits  = model(spec_in, training=False)
    probs   = tf.nn.softmax(logits[0]).numpy()
    conf    = float(np.max(probs))
    cid     = int(np.argmax(probs))
    label   = label_names[cid]

    if conf >= threshold:
        result = f'✅ {label}  ({conf:.1%} confidence)'
    else:
        result = (f"❓ Low confidence — best guess: '{label}' "
                  f'at {conf:.1%}  (threshold={threshold:.0%})')
    return result, waveform, conf, label


# ── demo on test batch ────────────────────────────────────────────────────────
print(f'Confidence threshold: {CONFIDENCE_THRESHOLD:.0%}\n')
n_above = n_below = 0

for spec_batch, label_batch in test_spectrogram_ds.take(1):
    for idx in range(min(20, len(label_batch))):
        true_label = label_names[label_batch[idx].numpy()]
        spec_in    = spec_batch[idx:idx+1]                    # (1,49,40)
        logits     = model(spec_in, training=False)
        probs      = tf.nn.softmax(logits[0]).numpy()
        conf       = float(np.max(probs))
        pred       = label_names[int(np.argmax(probs))]
        flag       = '✅' if conf >= CONFIDENCE_THRESHOLD else '❓'
        print(f'  True: {true_label:<12}  Pred: {pred:<12}  Conf: {conf:.1%}  {flag}')
        n_above += (1 if conf >= CONFIDENCE_THRESHOLD else 0)
        n_below += (0 if conf >= CONFIDENCE_THRESHOLD else 1)

print(f'\nAbove: {n_above}  |  Below: {n_below}')

result_text, audio_data, conf, pred = run_inference_with_threshold(
    data_dir / 'no/01bb6a2a_nohash_0.wav', model, label_names)
print(f'\n{result_text}')
display.display(display.Audio(audio_data, rate=16000))


### Display a confusion matrix

Use a [confusion matrix](https://developers.google.com/machine-learning/glossary#confusion-matrix) to check how well the model did classifying each of the commands in the test set:


In [ ]:
y_pred = model.predict(test_spectrogram_ds)

In [ ]:
y_pred = tf.argmax(y_pred, axis=1)

In [ ]:
y_true = tf.concat(list(test_spectrogram_ds.map(lambda s,lab: lab)), axis=0)

In [ ]:
confusion_mtx = tf.math.confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_mtx,
            xticklabels=label_names,
            yticklabels=label_names,
            annot=True, fmt='g')
plt.xlabel('Prediction')
plt.ylabel('Label')
plt.show()

## Run inference on an audio file

Finally, verify the model's prediction output using an input audio file of someone saying "no". How well does your model perform?

In [ ]:
raw      = tf.io.read_file(str(data_dir / 'no/01bb6a2a_nohash_0.wav'))
wf, _    = tf.audio.decode_wav(raw, desired_channels=1, desired_samples=16000)
waveform = tf.squeeze(wf, axis=-1)

spec     = waveform_to_spectrogram(waveform, quantize=False)   # (49,40) float32
spec_in  = tf.constant(spec[np.newaxis, ...], dtype=tf.float32)

prediction = model(spec_in)
plt.bar(label_names, tf.nn.softmax(prediction[0]))
plt.title('No')
plt.show()
display.display(display.Audio(waveform, rate=16000))


As the output suggests, your model should have recognized the audio command as "no".

## Export the model with preprocessing

The model's not very easy to use if you have to apply those preprocessing steps before passing data to the model for inference. So build an end-to-end version:

In [ ]:
class ExportModel(tf.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    @tf.function(input_signature=[
        tf.TensorSpec(
            shape=[TIME_STEPS, NUM_MEL_BINS],                    # ← (49, 40) no batch
            dtype=tf.float32,
            name='spectrogram'
        )
    ])
    def __call__(self, x):
        x      = tf.expand_dims(x, axis=0)                       # (49,40) → (1,49,40)
        logits = self.model(x, training=False)                   # (1, num_labels)
        probs  = tf.nn.softmax(logits)                           # (1, num_labels)
        return tf.squeeze(probs, axis=0)                         # (num_labels,) ✅



In [ ]:
DEFAULT_THRESHOLD = 0.7

# ── predict with imported SavedModel ──
def predict_with_threshold(model, spectrogram, label_names, threshold=DEFAULT_THRESHOLD):
    if hasattr(spectrogram, 'numpy'):
        spectrogram = spectrogram.numpy()
    spectrogram = np.squeeze(spectrogram)                        # (49, 40)
    spectrogram = tf.constant(spectrogram, dtype=tf.float32)

    # ── handle SavedModel (has signatures) vs ExportModel (direct call) ──
    if hasattr(model, 'signatures'):
        sig   = model.signatures['serving_default']
        out   = sig(spectrogram=spectrogram)
        probs = list(out.values())[0]
    else:
        probs = model(spectrogram)                               # ExportModel

    if hasattr(probs, 'numpy'):
        probs = probs.numpy()
    probs      = probs.flatten()
    best_index = int(np.argmax(probs))
    best_score = float(probs[best_index])
    return {
        'detected'    : best_score > threshold,
        'class_index' : best_index,
        'confidence'  : best_score,
        'class_name'  : str(label_names[best_index]) if best_score > threshold else None,
    }

Test run the "export" model:

In [ ]:
spectrogram = wav_to_spectrogram(str(data_dir / 'no/01bb6a2a_nohash_0.wav'))
export = ExportModel(model)
tf.saved_model.save(
    export, 'saved',
    signatures={
        'serving_default': export.__call__.get_concrete_function(
            tf.TensorSpec(shape=[TIME_STEPS, NUM_MEL_BINS],
                          dtype=tf.float32, name='spectrogram'))
    }
)
predict_with_threshold(export, spectrogram, label_names)


Save and reload the model, the reloaded model gives identical output:

In [ ]:
spectrogram = wav_to_spectrogram(str(data_dir / 'yes/004ae714_nohash_0.wav'))
print('Spectrogram shape:', spectrogram.shape)
imported = tf.saved_model.load('saved')
predict_with_threshold(imported, spectrogram, label_names)


In [ ]:
def generate_int8_tflite(model, dataset, output_path='micro_speech_quantized.tflite'):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    def representative_dataset():
        count = 0
        for spectrogram, _ in dataset:
            for i in range(spectrogram.shape[0]):
                if count >= 500:
                    return
                spec = spectrogram[i:i+1].numpy()               # (1, 49, 40) float32
                yield [spec.astype(np.float32)]
                count += 1

    converter.optimizations              = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset     = representative_dataset
    converter.target_spec.supported_ops  = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type       = tf.int8
    converter.inference_output_type      = tf.int8
    converter.experimental_new_quantizer = True

    tflite_model = converter.convert()
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    print(f'Saved → {output_path}  ({len(tflite_model)/1024:.1f} KB)')
    return tflite_model

tflite_model = generate_int8_tflite(model, train_spectrogram_ds)

In [ ]:
def predict_tflite(interpreter, spectrogram, label_names, threshold=0.6):
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    if hasattr(spectrogram, 'numpy'):
        spectrogram = spectrogram.numpy()
    spectrogram = np.squeeze(spectrogram).astype(np.float32)    # always float32 (49,40)
    spectrogram = spectrogram[np.newaxis, ...]                   # (1, 49, 40)

    if input_details[0]['dtype'] == np.int8:
        scale, zp   = input_details[0]['quantization']
        spectrogram = np.round(spectrogram / scale + zp).clip(-128, 127).astype(np.int8)
    # else: already float32, feed directly

    interpreter.set_tensor(input_details[0]['index'], spectrogram)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])

    if output_details[0]['dtype'] == np.int8:
        scale, zero = output_details[0]['quantization']
        logits = (output.astype(np.float32) - zero) * scale
    else:
        logits = output.astype(np.float32)

    logits     = logits.flatten()
    logits    -= np.max(logits)                                  # stable softmax
    probs      = np.exp(logits) / np.sum(np.exp(logits))
    class_id   = int(np.argmax(probs))
    confidence = float(probs[class_id])
    return {
        'detected'   : confidence > threshold,
        'class_id'   : class_id,
        'class_name' : str(label_names[class_id]) if confidence > threshold else None,
        'confidence' : confidence,
    }

## End-to-end TFLite pipeline

Allocate the classifier interpreter.  The preprocessor interpreters were
already allocated earlier (`interp_pre_i8`, `interp_pre_f32`).


In [ ]:
from ai_edge_litert.interpreter import Interpreter as LiteInterpreter

interp_cls = LiteInterpreter(model_path='micro_speech_quantized.tflite')
interp_cls.allocate_tensors()

_ci = interp_cls.get_input_details()[0]
_co = interp_cls.get_output_details()[0]
print(f'Classifier  in={_ci["dtype"].__name__}{list(_ci["shape"])}  '
      f'out={_co["dtype"].__name__}{list(_co["shape"])}')


### Two end-to-end paths

```
int16→int8  : WAV → int16 → interp_pre_i8  → int8 (49,40) → classifier → label
float32     : WAV → f32   → interp_pre_f32 → f32  (49,40) → classifier → label
```
Both paths call the **TFLM interpreter** for preprocessing — no Python DSP anywhere.


In [ ]:
def run_e2e_tflite(wav_path, interp_pre, interp_cls, label_names,
                   quantize=False, threshold=0.7):
    """
    Full MCU-equivalent pipeline through TFLM preprocessor + TFLite classifier.

    quantize=True  : int16 input → int8 preprocessor  → int8  (49,40)
    quantize=False : int16 input → f32  preprocessor  → f32   (49,40)
    Both feed into the int8 classifier.
    """
    waveform_np = wav_to_waveform(wav_path, quantize=False).numpy()

    # ── preprocessor (TFLM interpreter) ──────────────────────────────────────
    spec = _process_single(waveform_np, quantize=quantize)    # (49,40) i8/f32

    # ── classifier (standard TFLite interpreter) ──────────────────────────────
    cls_inp = interp_cls.get_input_details()[0]
    cls_out = interp_cls.get_output_details()[0]

    spec_f32 = spec.astype(np.float32)[np.newaxis, ...]       # (1,49,40) f32
    if cls_inp['dtype'] == np.int8:
        s, z    = cls_inp['quantization']
        spec_in = np.round(spec_f32 / s + z).clip(-128,127).astype(np.int8)
    else:
        spec_in = spec_f32

    interp_cls.set_tensor(cls_inp['index'], spec_in)
    interp_cls.invoke()
    output = interp_cls.get_tensor(cls_out['index'])

    if cls_out['dtype'] == np.int8:
        s, z   = cls_out['quantization']
        logits = (output.astype(np.float32) - z) * s
    else:
        logits = output.astype(np.float32)

    logits -= np.max(logits)
    probs   = np.exp(logits) / np.sum(np.exp(logits))
    probs   = probs.flatten()
    cid     = int(np.argmax(probs))
    conf    = float(probs[cid])
    return {
        'detected'  : conf >= threshold,
        'class_name': label_names[cid] if conf >= threshold else None,
        'class_id'  : cid,
        'confidence': conf,
    }


# ── verify both paths ─────────────────────────────────────────────────────────
for wav_path, expected in [
    (str(data_dir / 'no/01bb6a2a_nohash_0.wav'),  'no'),
    (str(data_dir / 'yes/004ae714_nohash_0.wav'), 'yes'),
]:
    for q, tag in [(False, 'float32  '), (True, 'int16→int8')]:
        pre = interp_pre_i8 if q else interp_pre_f32
        r   = run_e2e_tflite(wav_path, pre, interp_cls, label_names, quantize=q)
        ok  = '✅' if r['class_name'] == expected else '❌'
        print(f'{ok}  [{tag}]  expected={expected:<4}  '
              f'pred={str(r["class_name"]):<4}  conf={r["confidence"]:.1%}')


In [ ]:
# Install xxd if it is not available
MODEL_TFLITE = 'micro_speech_quantized.tflite'
MODEL_TFLITE_MICRO  = 'micro_speech_quantized.cc'
!apt-get update && apt-get -qq install xxd
# Convert to a C source file
!xxd -i {MODEL_TFLITE} > {MODEL_TFLITE_MICRO}
# Update variable names
REPLACE_TEXT = MODEL_TFLITE.replace('/', '_').replace('.', '_')
!sed -i 's/'{REPLACE_TEXT}'/g_model/g' {MODEL_TFLITE_MICRO}

In [ ]:
import zipfile
import os

zip_name = "dataset.zip"
base_dir = ""

files_to_zip = [
    "/kaggle/working/micro_speech_quantized.cc",
    "/kaggle/working/micro_speech_quantized.tflite",
    "/kaggle/working/saved/variables/variables.data-00000-of-00001",
    "/kaggle/working/saved/variables/variables.index",
    "/kaggle/working/saved/saved_model.pb",
    "/kaggle/working/saved/fingerprint.pb"
]

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        full_path = os.path.join(base_dir, file)
        z.write(full_path, arcname=file)

print("Zip created:", zip_name)

## Next steps

This tutorial demonstrated how to carry out simple audio classification/automatic speech recognition using a convolutional neural network with TensorFlow and Python. To learn more, consider the following resources:

- The [Sound classification with YAMNet](https://www.tensorflow.org/hub/tutorials/yamnet) tutorial shows how to use transfer learning for audio classification.
- The notebooks from [Kaggle's TensorFlow speech recognition challenge](https://www.kaggle.com/c/tensorflow-speech-recognition-challenge/overview).
- The
[TensorFlow.js - Audio recognition using transfer learning codelab](https://codelabs.developers.google.com/codelabs/tensorflowjs-audio-codelab/index.html#0) teaches how to build your own interactive web app for audio classification.
- [A tutorial on deep learning for music information retrieval](https://arxiv.org/abs/1709.04396) (Choi et al., 2017) on arXiv.
- TensorFlow also has additional support for [audio data preparation and augmentation](https://www.tensorflow.org/io/tutorials/audio) to help with your own audio-based projects.
- Consider using the [librosa](https://librosa.org/) library for music and audio analysis.

In [13]:
%cd /kaggle/working/
!rm -rf zephyr_tflm_v2_speech
!git clone --depth 1 --recurse-submodules https://github.com/williamchand/zephyr_tflm_v2_speech

/kaggle/working
Cloning into 'zephyr_tflm_v2_speech'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 27 (delta 1), reused 22 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 1.74 MiB | 9.60 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Submodule 'tflite_micro' (https://github.com/tensorflow/tflite-micro) registered for path 'tflite_micro'
Cloning into '/kaggle/working/zephyr_tflm_v2_speech/tflite_micro'...
remote: Enumerating objects: 24578, done.        
remote: Counting objects: 100% (339/339), done.        
remote: Compressing objects: 100% (161/161), done.        
remote: Total 24578 (delta 261), reused 178 (delta 178), pack-reused 24239 (from 4)        
Receiving objects: 100% (24578/24578), 34.37 MiB | 19.25 MiB/s, done.
Resolving deltas: 100% (17882/17882), done.
Submodule path 'tflite_micro': checked out 'f2b2b3f51cdf97fe190ef8bd6d1839013c9c179e'


In [14]:
%cd /kaggle/working/zephyr_tflm_v2_speech/
!python scripts/setup_links.py
%cd /kaggle/working/zephyr_tflm_v2_speech/tflite_micro/

/kaggle/working/zephyr_tflm_v2_speech
==> Setting up links into tflite_micro...

  linked:  micro_model_settings.h
  linked:  micro_speech_test.cc
  linked:  micro_speech_quantized.tflite
  linked:  train_micro_speech_model.ipynb

==> Done. You can now run:

  cd tflite_micro

  # Bazel
  bazel run tensorflow/lite/micro/examples/micro_speech:micro_speech_test

  # Make
  make -f tensorflow/lite/micro/tools/make/Makefile test_micro_speech_test

  # Make QEMU Cortex-M0 + CMSIS-NN
  make -f tensorflow/lite/micro/tools/make/Makefile \
    TARGET=cortex_m_qemu TARGET_ARCH=cortex-m0 \
    OPTIMIZED_KERNEL_DIR=cmsis_nn BUILD_TYPE=default \
    test_micro_speech_test
/kaggle/working/zephyr_tflm_v2_speech/tflite_micro


In [15]:
!make -f tensorflow/lite/micro/tools/make/Makefile test_micro_speech_test

Cloning into 'tensorflow/lite/micro/tools/make/downloads/pigweed'...
remote: Total 183201 (delta 112204), reused 183201 (delta 112204)
Receiving objects: 100% (183201/183201), 136.74 MiB | 27.71 MiB/s, done.
Resolving deltas: 100% (112204/112204), done.
Note: switching to '47268dff45019863e20438ca3746c6c62df6ef09'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 47268dff4 pw_hdlc_lite: Client I/O improvements
Switched to a new branch 'tflm'
Applying /kaggle/working/zephyr_tflm_v2_speech/tflite_micro/tensorflow/

In [16]:
!./gen/linux_x86_64_default_gcc/bin/micro_speech_test

[==========] Running tests.
[ RUN      ] MicroSpeechTest.NoTest
[       OK ] MicroSpeechTest.NoTest
[ RUN      ] MicroSpeechTest.YesTest
[       OK ] MicroSpeechTest.YesTest
[ RUN      ] MicroSpeechTest.SilenceTest
[       OK ] MicroSpeechTest.SilenceTest
[ RUN      ] MicroSpeechTest.NoiseTest
[       OK ] MicroSpeechTest.NoiseTest
[ RUN      ] MicroSpeechTest.RingBufferSuppression
[       OK ] MicroSpeechTest.RingBufferSuppression
[==========] 5 tests ran.
[  PASSED  ] 5 tests.
~~~ALL TESTS PASSED~~~
